Copyright 2026 Snowflake Inc.
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

# Module 1: Apache Iceberg Fundamentals

This notebook contains code examples for Module 1 videos.

⚠️ **IMPORTANT**: Before running cells in this notebook, open and run `Videos/Setup.ipynb` to download the required data.

This environment uses **Spark Connect** — a single Spark server runs in the background and all notebooks connect to it as lightweight clients. If you see `ConnectionRefusedError: [Errno 111] Connection refused` when initializing the Spark Session, the Spark Connect server may not be running. Check the Docker container logs with `docker logs jupyter-spark`. If the server failed to start, try restarting the container with `docker compose restart jupyter`.

## Setup

Initialize Spark session for Module 1 examples.

In [1]:
# Initialize Spark session for working with Iceberg tables

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Module1_Fundamentals") \
    .getOrCreate()

/opt/conda/lib/python3.13/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at spark/connect/base.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/conda/lib/python3.13/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at spark/connect/commands.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/conda/lib/python3.13/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at spark/connect/common.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/conda/lib/p

## Video: Course Introduction

## Video: What does it mean that Apache Iceberg is an Open Table Format?

## Video: The Open Lakehouse

In [2]:
# Display Spark configuration showing how to connect to Catalog (Polaris) and Storage (RustFS)
# Inside the spark-defaults.conf file, we have 
import os

spark_conf_path = os.path.expanduser('~/.sparkconf/spark-defaults.conf')

with open(spark_conf_path, 'r') as f:
    spark_config = f.read()
    print(spark_config)

# Copyright 2026 Snowflake Inc.
# SPDX-License-Identifier: Apache-2.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Iceberg Spark Dependencies
spark.jars.packages=org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.0,org.apache.iceberg:iceberg-aws-bundle:1.10.0,software.amazon.awssdk:bundle:2.20.18,software.amazon.awssdk:url-connection-client:2.20.18,org.apache.hadoop:hadoop-aws:3.4.1

# Iceberg Spark SQL/Catalyst Extensions
spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSpark

## Video: Modeling Data into an Apache Iceberg Table

In [3]:
# Create an unpartitioned Iceberg table from the NYC taxi Parquet files
# This demonstrates the basic table structure without partitioning

spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_unpartitioned
    USING iceberg
    TBLPROPERTIES (
        'format-version' = '3',
        'write.target-file-size-bytes' = '16777216'
    )
    AS SELECT * FROM parquet.`s3a://warehouse/raw/`
""")

DataFrame[]

In [4]:
# Query the first 3 columns to verify the table was created successfully

spark.sql("""
    SELECT 
        VendorID,
        tpep_pickup_datetime,
        tpep_dropoff_datetime
    FROM polaris.taxi.trips_unpartitioned
    LIMIT 10
""").show()

+--------+--------------------+---------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|
+--------+--------------------+---------------------+
|       1| 2023-07-01 00:29:59|  2023-07-01 00:40:15|
|       2| 2023-07-01 00:03:25|  2023-07-01 00:23:44|
|       2| 2023-07-01 00:38:29|  2023-07-01 00:48:53|
|       2| 2023-07-01 00:14:16|  2023-07-01 00:29:13|
|       1| 2023-07-01 00:11:15|  2023-07-01 00:20:47|
|       2| 2023-07-01 00:29:32|  2023-07-01 00:54:14|
|       2| 2023-07-01 00:24:43|  2023-07-01 00:47:49|
|       2| 2023-07-01 00:35:28|  2023-07-01 00:45:30|
|       1| 2023-07-01 00:31:05|  2023-07-01 00:35:20|
|       2| 2023-07-01 00:38:00|  2023-07-01 00:54:47|
+--------+--------------------+---------------------+



### View Table Files in RustFS

[Open trips_unpartitioned in RustFS](http://localhost:9001/browser/warehouse/taxi/trips_unpartitioned/)

Login: `admin` / `password`

In [5]:
# Use the .files metadata table to inspect data files and aggregated table statistics

spark.sql("""
    SELECT 
        SUBSTRING(file_path, POSITION('/data/' IN file_path)) as file_path,
        file_size_in_bytes / 1024 / 1024 as size_mb,
        record_count
    FROM polaris.taxi.trips_unpartitioned.files
""").show(truncate=False)

spark.sql("""
    SELECT 
        COUNT(*) as total_files,
        SUM(record_count) as total_records,
        ROUND(SUM(file_size_in_bytes) / 1024 / 1024, 2) as total_size_mb
    FROM polaris.taxi.trips_unpartitioned.files
""").show()

+-------------------------------------------------------------------+-------------------+------------+
|file_path                                                          |size_mb            |record_count|
+-------------------------------------------------------------------+-------------------+------------+
|/data/00000-25-83001fac-d39a-4671-ac58-ba8d3d3e00a8-0-00001.parquet|16.06031608581543  |1059000     |
|/data/00000-25-83001fac-d39a-4671-ac58-ba8d3d3e00a8-0-00002.parquet|15.952461242675781 |1059000     |
|/data/00000-25-83001fac-d39a-4671-ac58-ba8d3d3e00a8-0-00003.parquet|16.100919723510742 |1058000     |
|/data/00000-25-83001fac-d39a-4671-ac58-ba8d3d3e00a8-0-00004.parquet|2.4592323303222656 |131234      |
|/data/00001-26-83001fac-d39a-4671-ac58-ba8d3d3e00a8-0-00001.parquet|16.12346076965332  |1038000     |
|/data/00001-26-83001fac-d39a-4671-ac58-ba8d3d3e00a8-0-00002.parquet|16.11240005493164  |1058000     |
|/data/00001-26-83001fac-d39a-4671-ac58-ba8d3d3e00a8-0-00003.parquet|12.5

## Video: Hidden Partitioning in Apache Iceberg Tables


In [6]:
# Create a table with hidden partitioning using the days() transform
# Iceberg automatically partitions by day without requiring a separate day column

spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_by_day
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    TBLPROPERTIES (
        'format-version' = '3',
        'write.target-file-size-bytes' = '16777216'
    )
    AS SELECT * FROM parquet.`s3a://warehouse/raw/`
""")

DataFrame[]

In [7]:
# Query with timestamp filter - Iceberg automatically applies partition pruning
# Check Spark UI (http://localhost:4040/SQL/) to see which files were scanned vs. skipped

spark.sql("""
    SELECT 
        COUNT(*) as total_trips,
        ROUND(AVG(total_amount), 2) as avg_fare,
        MIN(tpep_pickup_datetime) as first_pickup,
        MAX(tpep_pickup_datetime) as last_pickup
    FROM polaris.taxi.trips_by_day
    WHERE tpep_pickup_datetime >= '2023-08-01'
      AND tpep_pickup_datetime < '2023-09-01'
""").show()

+-----------+--------+-------------------+-------------------+
|total_trips|avg_fare|       first_pickup|        last_pickup|
+-----------+--------+-------------------+-------------------+
|    2824199|   28.63|2023-08-01 00:00:00|2023-08-31 23:59:59|
+-----------+--------+-------------------+-------------------+



### View Query Execution in Spark UI

[Open Spark SQL Tab](http://localhost:4040/SQL/)

Check the query metrics to see:
- **Number of result data files**: Only files for the queried date range
- **Number of skipped data files**: Files from other partitions that were pruned
- **Partition pruning**: Iceberg automatically applied the partition filter

In [8]:
# View column metrics (min/max bounds) stored in manifest files for predicate pushdown
# These statistics enable file skipping without opening the actual data files

spark.sql("""
    SELECT 
        SUBSTRING(file_path, POSITION('day=' IN file_path)) as file_path,
        partition,
        record_count,
        ROUND(file_size_in_bytes / 1024 / 1024, 2) as size_mb,
        readable_metrics.PULocationID.lower_bound as PULocationID_min,
        readable_metrics.PULocationID.upper_bound as PULocationID_max,
        readable_metrics.total_amount.lower_bound as total_amount_min,
        readable_metrics.total_amount.upper_bound as total_amount_max
    FROM polaris.taxi.trips_by_day.files
    ORDER BY partition DESC
    LIMIT 10
""").show(truncate=False)

+----------------------------------------------------------------------------+------------+------------+-------+----------------+----------------+----------------+----------------+
|file_path                                                                   |partition   |record_count|size_mb|PULocationID_min|PULocationID_max|total_amount_min|total_amount_max|
+----------------------------------------------------------------------------+------------+------------+-------+----------------+----------------+----------------+----------------+
|day=2023-11-23/00005-55-351a6c96-9eec-4bd0-805b-4adf93a5818a-0-00013.parquet|{2023-11-23}|2           |0.01   |132             |142             |36.12           |72.35           |
|day=2023-11-02/00008-58-351a6c96-9eec-4bd0-805b-4adf93a5818a-0-00014.parquet|{2023-11-02}|3           |0.01   |65              |233             |13.6            |101.5           |
|day=2023-11-01/00007-57-351a6c96-9eec-4bd0-805b-4adf93a5818a-0-00009.parquet|{2023-11-01}|8   

## Video: Summary of Module 1